# Quantile Forecasting & Safety Stock — Smooth Demand

Usa os **melhores parâmetros já encontrados pelo Optuna** (carregados do `metadata.json`),
trocando apenas o `objective` de `regression` para `quantile`.

| Quantil | Uso |
|---|---|
| q50 | Previsão pontual (mediana) |
| q80 | Estoque moderado |
| q95 | Estoque conservador / safety stock |

**Safety Stock** = q95 − q50 (buffer acima da mediana para garantir nível de serviço)

In [ ]:
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

from mlforecast import MLForecast
from mlforecast.target_transforms import LocalStandardScaler
from mlforecast.lag_transforms import RollingMean
from lightgbm import LGBMRegressor

import sys
sys.path.insert(0, '../../src')
from config import LoadConfig
from pipeline.features import FeatureLoader, HOLIDAY_COLS, WEATHER_COLS, STATIC_COLS

warnings.filterwarnings('ignore')
os.makedirs('../../plots', exist_ok=True)

cfg    = LoadConfig()
loader = FeatureLoader(cfg)

## 1. Carrega melhores params do artefato treinado

In [ ]:
ARTIFACT_DIR = '../../artifacts/models/smooth'
with open(f'{ARTIFACT_DIR}/metadata.json') as f:
    meta = json.load(f)

BEST_PARAMS = meta['best_params']
FREQ        = meta['freq']
HORIZON     = meta['horizon']
TARGET_COL  = 'y'
SEED        = 42

print(f'Freq    : {FREQ}')
print(f'Horizon : {HORIZON} semanas')
print(f'Params  : {BEST_PARAMS}')

## 2. Carrega dados — mesmo split do model_building

In [ ]:
df = loader.build_dataset('smooth')

cutoff   = df['ds'].max() - pd.Timedelta(weeks=HORIZON)
df_train = df[df['ds'] <= cutoff].copy()
df_test  = df[df['ds'] >  cutoff].copy()

STATIC_FEATURES = [c for c in STATIC_COLS if c in df_train.columns]

print(f'Cutoff : {cutoff.date()}')
print(f'Train  : {len(df_train):,} rows | {df_train["unique_id"].nunique():,} séries')
print(f'Test   : {len(df_test):,} rows')

## 3. Configura Quantile LightGBM — q50 / q80 / q95

Subclasses com nomes distintos para que o MLForecast gere colunas separadas no output.

In [ ]:
LAG_PRESETS = {
    'short':  [4, 13],
    'medium': [4, 8, 13, 26],
    'long':   [4, 8, 13, 26, 52],
}
LAG_TF_MAP = {
    'rolling_mean_4':  RollingMean(window_size=4),
    'rolling_mean_13': RollingMean(window_size=13),
    'none':            None,
}

lags = LAG_PRESETS[BEST_PARAMS['lag_preset']]
lag_tf = LAG_TF_MAP[BEST_PARAMS['lag_transform']]
lag_transforms = {lag: [lag_tf] for lag in lags} if lag_tf else None

# Parâmetros base (remove chaves de estrutura, mantém só LightGBM)
lgbm_base = {
    k: v for k, v in BEST_PARAMS.items()
    if k not in ('lag_preset', 'lag_transform', 'target_transform')
}

# Subclasses para gerar colunas distintas no MLForecast
class LGBMq50(LGBMRegressor): pass
class LGBMq80(LGBMRegressor): pass
class LGBMq95(LGBMRegressor): pass

QUANTILES = [
    (LGBMq50, 0.50),
    (LGBMq80, 0.80),
    (LGBMq95, 0.95),
]

models = [
    cls(objective='quantile', alpha=alpha, random_state=SEED, verbose=-1, **lgbm_base)
    for cls, alpha in QUANTILES
]

mlf = MLForecast(
    models=models,
    freq=FREQ,
    lags=lags,
    lag_transforms=lag_transforms,
    target_transforms=[LocalStandardScaler()],
    date_features=['month', 'quarter', 'week'],
)

print(f'Lags    : {lags}')
print(f'Lag tf  : {BEST_PARAMS["lag_transform"]}')
print(f'Modelos : {[type(m).__name__ for m in models]}')

## 4. Treino e predição no test set

In [ ]:
mlf.fit(df_train, static_features=STATIC_FEATURES)
print('Fit concluído')

In [ ]:
df_static   = loader.df_static
df_holidays = loader.df_holidays
df_weather  = loader.df_weather

avail_hol = [c for c in HOLIDAY_COLS if c in df_train.columns]
avail_wth = [c for c in WEATHER_COLS if c in df_train.columns]

future = (
    mlf.make_future_dataframe(h=HORIZON)
    .merge(df_static[['unique_id', 'region_id']], on='unique_id', how='left')
)
if avail_hol and df_holidays is not None:
    future = future.merge(
        df_holidays[['ds', 'region_id'] + avail_hol], on=['ds', 'region_id'], how='left'
    )
if avail_wth and df_weather is not None:
    future = future.merge(
        df_weather[['ds', 'region_id'] + avail_wth], on=['ds', 'region_id'], how='left'
    )

preds = mlf.predict(h=HORIZON, X_df=future.drop(columns='region_id'))

results = (
    df_test[['unique_id', 'ds', TARGET_COL]]
    .merge(preds, on=['unique_id', 'ds'], how='left')
)
results['safety_stock'] = results['LGBMq95'] - results['LGBMq50']

results.head()

## 5. Métricas de cobertura e Pinball Loss

**Coverage**: % das semanas em que o real ficou abaixo do quantil previsto (target = o próprio α).  
**Pinball Loss**: métrica assimétrica — penaliza mais quando o real ultrapassa o quantil.

In [ ]:
def pinball_loss(y_true, y_pred, alpha):
    err = y_true - y_pred
    return float(np.mean(np.where(err >= 0, alpha * err, (alpha - 1) * err)))

q_cols  = ['LGBMq50', 'LGBMq80', 'LGBMq95']
alphas  = [0.50, 0.80, 0.95]

print(f'{"Modelo":<12} {"α":>5}  {"Pinball":>10}  {"Coverage":>10}  {"Target":>8}')
print('-' * 55)
for col, alpha in zip(q_cols, alphas):
    pl  = pinball_loss(results[TARGET_COL].values, results[col].values, alpha)
    cov = (results[TARGET_COL] <= results[col]).mean()
    print(f'{col:<12} {alpha:>5.2f}  {pl:>10.4f}  {cov:>9.1%}   {alpha:>7.0%}')

print(f'\nSafety Stock médio : {results["safety_stock"].mean():.1f} unidades/série/semana')
print(f'Safety Stock mediano: {results["safety_stock"].median():.1f}')

## 6. Visualização — Séries por cobertura do q95

In [ ]:
N_HIST = 52

coverage_by_series = (
    results.groupby('unique_id')
    .apply(lambda g: (g[TARGET_COL] <= g['LGBMq95']).mean())
    .reset_index(name='coverage_q95')
    .sort_values('coverage_q95')
    .reset_index(drop=True)
)
n = len(coverage_by_series)

selected = pd.concat([
    coverage_by_series.tail(2).assign(quality='Alta cobertura',   color='seagreen'),
    coverage_by_series.iloc[[n // 2]].assign(quality='Mediana',   color='steelblue'),
    coverage_by_series.head(2).assign(quality='Baixa cobertura',  color='tomato'),
], ignore_index=True)

print(selected[['unique_id', 'coverage_q95', 'quality']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(26, 5))

col_titles = [
    'Alta cobertura #1', 'Alta cobertura #2',
    'Mediana',
    'Baixa cobertura #1', 'Baixa cobertura #2',
]

for ax_idx, (_, row) in enumerate(selected.iterrows()):
    ax  = axes[ax_idx]
    uid = row['unique_id']
    cov = row['coverage_q95']
    c   = row['color']

    hist   = df_train[df_train['unique_id'] == uid].sort_values('ds').tail(N_HIST)
    actual = df_test[df_test['unique_id'] == uid].sort_values('ds')
    pred   = results[results['unique_id'] == uid].sort_values('ds')
    cutoff = hist['ds'].max()

    # ── Bandas quantílicas ────────────────────────────────────────────────────
    ax.fill_between(pred['ds'], pred['LGBMq80'], pred['LGBMq95'],
                    alpha=0.20, color=c, label='q80–q95')
    ax.fill_between(pred['ds'], pred['LGBMq50'], pred['LGBMq80'],
                    alpha=0.35, color=c, label='q50–q80')

    # ── Séries ────────────────────────────────────────────────────────────────
    ax.plot(hist['ds'], hist[TARGET_COL],
            color='#bbbbbb', linewidth=1.4, label='Histórico')
    ax.plot([cutoff, actual['ds'].iloc[0]],
            [hist[TARGET_COL].iloc[-1], actual[TARGET_COL].iloc[0]],
            color='#bbbbbb', linewidth=1.4, linestyle='--')
    ax.plot(actual['ds'], actual[TARGET_COL],
            color='#2c7bb6', linewidth=2, marker='o', markersize=5, label='Real')
    ax.plot(pred['ds'], pred['LGBMq50'],
            color=c, linewidth=2, marker='s', markersize=4,
            linestyle='--', label='q50 (ponto)')
    ax.plot(pred['ds'], pred['LGBMq95'],
            color=c, linewidth=1.2, marker='^', markersize=4,
            linestyle=':', label='q95 (safety)')

    ax.axvline(cutoff, color='black', linestyle=':', linewidth=0.8, alpha=0.5)

    ss_mean = pred['safety_stock'].mean()
    ax.set_title(f'{col_titles[ax_idx]}\nCoverage q95={cov:.0%}  |  SS={ss_mean:.0f}u',
                 fontsize=8, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%y'))
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
    ax.grid(axis='y', alpha=0.25)

    if ax_idx == 0:
        ax.legend(fontsize=7, loc='upper left')

fig.suptitle('Quantile Forecasting — Safety Stock (Smooth Demand)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../../plots/quantile_safe_stock.png', dpi=130, bbox_inches='tight')
plt.show()

## 7. Distribuição do Safety Stock por série

In [ ]:
ss_by_series = results.groupby('unique_id')['safety_stock'].mean()
ss_weekly    = results.groupby('ds')['safety_stock'].agg(['mean', 'median'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Histograma ────────────────────────────────────────────────────────────────
axes[0].hist(ss_by_series, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
for p, color, label in [(0.50, 'green', 'p50'), (0.75, 'orange', 'p75'), (0.95, 'red', 'p95')]:
    val = ss_by_series.quantile(p)
    axes[0].axvline(val, color=color, linestyle='--', linewidth=1.5,
                    label=f'{label} = {val:.0f}u')
axes[0].set_title('Distribuição do Safety Stock médio por série', fontweight='bold')
axes[0].set_xlabel('Unidades  (q95 − q50)')
axes[0].set_ylabel('Nº de séries')
axes[0].legend(fontsize=9)

# ── Por semana do horizonte ───────────────────────────────────────────────────
x = range(len(ss_weekly))
axes[1].bar(x, ss_weekly['mean'], color='steelblue', alpha=0.7, label='Média')
axes[1].plot(x, ss_weekly['median'], color='orange', marker='o',
             linewidth=2, markersize=6, label='Mediana')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'H+{i+1}' for i in x])
axes[1].set_title('Safety Stock por semana do horizonte', fontweight='bold')
axes[1].set_ylabel('Unidades')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../../plots/safe_stock_distribution.png', dpi=130, bbox_inches='tight')
plt.show()

print('\nSafety Stock por série (unidades):')
print(ss_by_series.describe().round(1).to_string())